# 01 - Data Cleaning (PuneBite Analyzer)

**Objective:** turn the raw Zomato Pune file into a clean, analysis-ready dataset.
**Input:** `data/raw/*.csv` (12,189 rows x 104 columns)
**Output:** `data/processed/restaurants_clean.csv`
**Syllabus link:** Unit II (data cleaning, missing values, outliers, duplicates, feature engineering); Lab 1.

All cleaning logic lives in `ml/cleaning.py`; this notebook runs it step by step and shows what each step changes.

In [ ]:
import sys, os, glob
sys.path.append(os.path.abspath(".."))      # so we can import ml/cleaning.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ml import cleaning as c

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

RAW_PATH = glob.glob("../data/raw/*.csv")[0]
print("Using:", RAW_PATH)

## 1. Load and inspect the raw data

In [ ]:
raw = c.load_raw(RAW_PATH)
print("Shape:", raw.shape)
raw.iloc[:, :17].head()

In [ ]:
# first 17 columns are descriptive; the rest are yes/no amenity columns + spam_review
print(list(raw.columns[:17]))
print("Amenity columns:", len(raw.columns) - 18, "| last column:", raw.columns[-1])
raw.iloc[:, :17].dtypes

## 2. Problems in the raw data (before cleaning)

We measure each problem first, so we can prove afterwards that it is fixed.

In [ ]:
rating_num = pd.to_numeric(raw["Ratings_out_of_5"], errors="coerce")
votes_num = pd.to_numeric(raw["Number of votes"].astype(str).str.extract(r"(\d+)")[0])
cost_num = pd.to_numeric(raw["Charges_for_two"].astype(str).str.replace(r"[^\d]", "", regex=True), errors="coerce")

problems = pd.Series({
    "Rating is '-' (unrated)": int(rating_num.isna().sum()),
    "Rating is 0.0 (placeholder)": int((rating_num == 0).sum()),
    "Votes == 0": int((votes_num == 0).sum()),
    "Price 'Not Present'": int(cost_num.isna().sum()),
    "Max price (Rs)": int(cost_num.max()),
    "Median price (Rs)": int(cost_num.median()),
    "Duplicate rows": int(raw.duplicated().sum()),
    "Duplicate Web_Link": int(raw["Web_Link"].duplicated().sum()),
})
problems

In [ ]:
# Are the 0.0 ratings real scores? Check their votes.
print("Votes of restaurants with rating 0.0:", votes_num[rating_num == 0].describe()[["count", "min", "max"]].to_dict())

**Finding:** every restaurant with rating `0.0` also has `0` votes, so `0.0` is a placeholder for "no reviews yet", **not** a real score. Treating it as a real rating would drag the average down badly. We convert both `-` and `0.0` to missing (NaN).

## 3. Cleaning, step by step
### 3.1 Rename columns
`Sponsored` actually holds the restaurant type, so it becomes `establishment_type`. Amenity names become snake_case.

In [ ]:
df, amenity_cols = c.rename_columns(raw)
print(len(amenity_cols), "amenity columns, e.g.", amenity_cols[:6])
df[["name", "locality", "establishment_type", "rating", "votes"]].head()

### 3.2 Remove duplicates

In [ ]:
before = len(df)
df = c.drop_duplicates(df)
print(f"Rows: {before} -> {len(df)}  (removed {before - len(df)})")

### 3.3 Votes, rating and text columns
`"7029  votes"` becomes the integer `7029`. Unrated `-` and placeholder `0.0` ratings become NaN, and a `has_rating` flag is added.

In [ ]:
df = c.clean_text_columns(df)
df = c.clean_votes(df)
df = c.clean_rating(df)

print("Rated restaurants:", df["has_rating"].sum(), "| unrated:", (~df["has_rating"]).sum())
df[["name", "rating", "votes", "has_rating"]].head()

### 3.4 Price (`cost_for_two`)
`"Rs 1,400"` becomes `1400`. Missing prices are imputed with the **median of the same locality + restaurant type** (a domain-based method), falling back to the type median, then the overall median. Extreme prices are capped at the 99th percentile in `cost_capped`; the raw value stays in `cost_for_two`. Median is used instead of mean because it isn't pulled by outliers.

In [ ]:
df = c.clean_cost(df)
print("Missing prices imputed:", int(df["cost_missing"].sum()))
print("Raw max:", df["cost_for_two"].max(), "| capped max:", df["cost_capped"].max())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.boxplot(x=df["cost_for_two"], ax=ax[0]); ax[0].set_title("Price - raw (outlier visible)")
sns.boxplot(x=df["cost_capped"], ax=ax[1]); ax[1].set_title("Price - capped at 99th percentile")
plt.tight_layout(); plt.show()

### 3.5 Review star percentages
`"79%"` becomes `79.0`. Restaurants with no reviews get NaN (not fake zeros). We also add three helper features: `positive_pct` (4 + 5 stars), `extreme_pct` (5 + 1 stars) and `polarization_pct` (high only when both 5-star and 1-star shares are large, meaning people love it or hate it).

> These columns are what the rating is *built from*. Use them for analysis only, never as inputs when predicting the rating (data leakage).

In [ ]:
df = c.clean_star_percentages(df)
df[["name", "rating", "star5_pct", "star1_pct", "positive_pct", "polarization_pct"]].head()

### 3.6 Cuisines, restaurant type, amenities, payments
Cuisines and amenities are stored as `|`-separated text (easy to load into MongoDB later). We also create counts and the primary cuisine/type for modelling.

In [ ]:
df = c.parse_list_columns(df, amenity_cols)
df = c.add_payment_flags(df)
df, amenity_cols, dropped = c.drop_constant_amenities(df, amenity_cols)
df = df.drop(columns=["phone"])

print("Constant amenity columns dropped:", dropped)
print("Amenity columns kept:", len(amenity_cols))
df[["name", "cuisines", "n_cuisines", "primary_cuisine", "primary_type", "n_amenities"]].head()

## 4. Validation after cleaning

In [ ]:
assert df["url"].duplicated().sum() == 0
assert (df["rating"].dropna() > 0).all()
assert df["votes"].dtype.kind == "i"

print("Final shape:", df.shape)
na = df.isna().sum()
na[na > 0]

In [ ]:
# Lab 1 style summary statistics
stats = df[["rating", "votes", "cost_capped"]].agg(["count", "mean", "median", "std", "var", "min", "max"]).T
stats.round(2)

## 5. Quick visual check

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["rating"].dropna(), bins=20, kde=True, ax=ax[0])
ax[0].set_title("Distribution of ratings (rated restaurants)")

top = df["primary_type"].value_counts()
pie = pd.concat([top.head(6), pd.Series({"Other": top.iloc[6:].sum()})])
ax[1].pie(pie, labels=pie.index, autopct="%1.0f%%", startangle=90)
ax[1].set_title("Restaurant types")

plt.tight_layout(); plt.show()

In [ ]:
r = df["rating"].dropna()
print(f"Rated restaurants: {len(r)}  |  rating >= 4.0: {(r >= 4).mean():.1%}  |  skew: {r.skew():.2f}")

**Observation:** ratings cluster around 3.4 and only about 1 in 9 rated restaurants reaches 4.0 or above. Our later classification target (`rating >= 4.0`) is therefore **imbalanced**, so we will use F1 and ROC-AUC, not accuracy alone.

## 6. Save the cleaned dataset

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
OUT = "../data/processed/restaurants_clean.csv"
df.to_csv(OUT, index=False)
print("Saved:", OUT, df.shape)

## 7. Notes for the report (fill in)

- Rows removed as duplicates: ____
- Unrated restaurants (kept for EDA, excluded from rating models): ____
- Price imputation method and number imputed: ____
- Price outlier handling: ____
- Biggest finding in this step: ____